In [1]:
import json
import pandas as pd

df = pd.read_csv(
    "all_proteomes_merged.tsv",
    sep="\t",          # TSV 的关键
    header=0,          # 第一行是列名（默认）
    encoding="utf-8",  # 常见编码：utf-8 / utf-8-sig / gbk
)

df.head()  # 查看前几行数据

,primaryAccession,uniProtkbId,organism,shortOrganism,entryType,proteinName,nameType,evidenceCode,evidenceSource,evidenceId,comment
0,A5A616,MGTS_ECOLI,Escherichia coli (strain K12),Escherichia_coli,UniProtKB reviewed (Swiss-Prot),Small protein MgtS,recommended,ECO:0000305,NaN,NaN,FUNCTION : Modulates intracellular Mg(2+) leve...
1,O32583,THIS_ECOLI,Escherichia coli (strain K12),Escherichia_coli,UniProtKB reviewed (Swiss-Prot),Sulfur carrier protein ThiS,recommended,NaN,NaN,NaN,FUNCTION : Is the sulfur donor in the synthesi...
2,O32583,THIS_ECOLI,Escherichia coli (strain K12),Escherichia_coli,UniProtKB reviewed (Swiss-Prot),Thiamine biosynthesis protein ThiS,alternative,NaN,NaN,NaN,FUNCTION : Is the sulfur donor in the synthesi...
3,P00350,6PGD_ECOLI,Escherichia coli (strain K12),Escherichia_coli,UniProtKB reviewed (Swiss-Prot),"6-phosphogluconate dehydrogenase, decarboxylating",recommended,NaN,NaN,NaN,FUNCTION : Catalyzes the oxidative decarboxyla...
4,P00363,FRDA_ECOLI,Escherichia coli (strain K12),Escherichia_coli,UniProtKB reviewed (Swiss-Prot),Fumarate reductase flavoprotein subunit,recommended,NaN,NaN,NaN,"FUNCTION : Two distinct, membrane-bound, FAD-c..."


In [2]:
# 查看列名和数据
print("列名:", df.columns.tolist())
print("\n查看前几行的 proteinName:")
print(df[['proteinName']].head(10))

#查看有多少sample
print(df)

列名: ['primaryAccession', 'uniProtkbId', 'organism', 'shortOrganism', 'entryType', 'proteinName', 'nameType', 'evidenceCode', 'evidenceSource', 'evidenceId', 'comment']

查看前几行的 proteinName:
                                         proteinName
0                                 Small protein MgtS
1                        Sulfur carrier protein ThiS
2                 Thiamine biosynthesis protein ThiS
3  6-phosphogluconate dehydrogenase, decarboxylating
4            Fumarate reductase flavoprotein subunit
5     Quinol-fumarate reductase flavoprotein subunit
6              NADP-specific glutamate dehydrogenase
7                Type II NADH:quinone oxidoreductase
8                                   Cupric reductase
9                               NADH dehydrogenase-2
       primaryAccession  uniProtkbId                       organism  \
0                A5A616   MGTS_ECOLI  Escherichia coli (strain K12)   
1                O32583   THIS_ECOLI  Escherichia coli (strain K12)   
2              

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
import json
from datasets import Dataset, DatasetDict
import numpy as np

# ============= 数据清洗和预处理 =============
print("="*80)
print("开始数据清洗和预处理")
print("="*80)

# 1. 删除缺失值
df = df.dropna(subset=["uniProtkbId", "proteinName", "nameType"])
print(f"删除缺失值后: {len(df)} 条记录")

# 2. 只保留 recommended 类型
df = df[df["nameType"].str.lower() == "recommended"].copy()
print(f"只保留 recommended 后: {len(df)} 条记录")

# 3. 高效检测 comment 泄露 product_name（使用向量化操作，比 apply 快 10x+）
def vectorized_leak_check(df):
    """向量化泄露检测，比逐行 apply 快得多"""
    protein_names = df['proteinName'].fillna('').str.lower()
    comments = df['comment'].fillna('').str.lower()
    # 使用 numpy 向量化操作
    leak_mask = np.array([
        pn in c if pn and c else False 
        for pn, c in zip(protein_names, comments)
    ])
    return leak_mask

leak_mask = vectorized_leak_check(df)
n_leaked = leak_mask.sum()
if n_leaked > 0:
    print(f"剔除 {n_leaked} 个 comment 泄露 product_name 的样本")
    df = df[~leak_mask].copy()
    
# 4. 创建规范化的列名
df["NAME"] = df["uniProtkbId"].astype(str).str.strip()
df["PRODUCT_NAME"] = df["proteinName"].astype(str).str.strip()

# 5. 去重
original_len = len(df)
df = df.drop_duplicates(subset=["NAME", "PRODUCT_NAME"])
print(f"去重后: {len(df)} 条记录 (删除 {original_len - len(df)} 条重复)")

# ============= 按 organism 分割数据集 =============
print("\n" + "="*80)
print("按 organism 分割数据集（确保不同集合的 organism 完全不重叠）")
print("="*80)

group_col = "shortOrganism" if "shortOrganism" in df.columns else "organism"
groups = df[group_col].astype(str)

# 使用 GroupShuffleSplit 确保同一 organism 的所有样本在同一集合
gss = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
train_idx, temp_idx = next(gss.split(df, groups=groups))
train = df.iloc[train_idx].copy()
temp = df.iloc[temp_idx].copy()

# dev/test 再按 group 切
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
groups_temp = temp[group_col].astype(str)
dev_idx, test_idx = next(gss2.split(temp, groups=groups_temp))
dev = temp.iloc[dev_idx].copy()
test = temp.iloc[test_idx].copy()

# 验证 organism 不重叠
train_organisms = set(train[group_col].unique())
dev_organisms = set(dev[group_col].unique())
test_organisms = set(test[group_col].unique())

print(f"\n训练集 organisms 数量: {len(train_organisms)}, 样本数: {len(train)}")
print(f"验证集 organisms 数量: {len(dev_organisms)}, 样本数: {len(dev)}")
print(f"测试集 organisms 数量: {len(test_organisms)}, 样本数: {len(test)}")

# 检查重叠
overlaps = [
    ("训练集 ∩ 验证集", train_organisms & dev_organisms),
    ("训练集 ∩ 测试集", train_organisms & test_organisms),
    ("验证集 ∩ 测试集", dev_organisms & test_organisms),
]
print(f"\n验证 organism 不重叠:")
all_clean = True
for name, overlap in overlaps:
    print(f"  {name}: {len(overlap)} 个 organisms")
    if overlap:
        all_clean = False
        
if all_clean:
    print("\n✓ 确认: 训练集、验证集、测试集的 organisms 完全不重叠!")
else:
    print("\n⚠️ 警告: 发现 organism 重叠!")

# ============= 生成 3 个阶梯级别的数据集 =============
print("\n" + "="*80)
print("生成 3 个阶梯级别的数据集")
print("="*80)

# 创建采样子集（用于 dev/test）
dev_subset = dev.sample(n=min(10000, len(dev)), random_state=42).reset_index(drop=True)
test_subset = test.sample(n=min(10000, len(test)), random_state=42).reset_index(drop=True)

def create_and_save_dataset(train_df, dev_df, test_df, columns, save_path, level_name):
    """统一的数据集创建和保存函数"""
    # 处理缺失值（主要是 comment 字段）
    for col in columns:
        if col == "comment":
            train_df = train_df.copy()
            dev_df = dev_df.copy()
            test_df = test_df.copy()
            train_df["comment"] = train_df["comment"].fillna("")
            dev_df["comment"] = dev_df["comment"].fillna("")
            test_df["comment"] = test_df["comment"].fillna("")
    
    train_ds = Dataset.from_pandas(train_df[columns], preserve_index=False)
    dev_ds = Dataset.from_pandas(dev_df[columns], preserve_index=False)
    test_ds = Dataset.from_pandas(test_df[columns], preserve_index=False)
    
    dataset_dict = DatasetDict({
        "train": train_ds,
        "dev": dev_ds,
        "test": test_ds
    })
    dataset_dict.save_to_disk(save_path)
    print(f"[{level_name}] Train: {len(train_ds)}, Dev: {len(dev_ds)}, Test: {len(test_ds)}")
    print(f"  已保存到: {save_path}/")
    return dataset_dict

# Level 1: NAME, PRODUCT_NAME
print("\n[Dataset 1] 字段: NAME, PRODUCT_NAME")
dataset_dict_1 = create_and_save_dataset(
    train, dev, test, 
    ["NAME", "PRODUCT_NAME"], 
    "protein_dataset_level1_full", "Level 1"
)

# Level 2: NAME, organism, PRODUCT_NAME
print("\n[Dataset 2] 字段: NAME, organism, PRODUCT_NAME")
dataset_dict_2 = create_and_save_dataset(
    train, dev, test,
    ["NAME", "organism", "PRODUCT_NAME"],
    "protein_dataset_level2_full", "Level 2"
)

# Level 3: NAME, organism, comment, PRODUCT_NAME (使用采样的 dev/test)
print("\n[Dataset 3] 字段: NAME, organism, comment, PRODUCT_NAME")
dataset_dict_3 = create_and_save_dataset(
    train, dev_subset, test_subset,
    ["NAME", "organism", "comment", "PRODUCT_NAME"],
    "protein_dataset_level3_full", "Level 3"
)

print("\n" + "="*80)
print("✓ 所有3个数据集生成完成！")
print("="*80)

In [31]:
# 加载并验证三个数据集
from datasets import load_from_disk

print("\n" + "="*80)
print("加载并验证三个数据集")
print("="*80)



# 加载 Dataset 2
loaded_ds2 = load_from_disk("protein_dataset_level2")
print("\n[Dataset 2] 字段: NAME, organism, PRODUCT_NAME")
print(f"  总体结构: {loaded_ds2}")
print(f"  Train 示例:")
print(f"    {loaded_ds2['train'][0]}")





加载并验证三个数据集

[Dataset 2] 字段: NAME, organism, PRODUCT_NAME
  总体结构: DatasetDict({
    train: Dataset({
        features: ['NAME', 'organism', 'PRODUCT_NAME'],
        num_rows: 70000
    })
    dev: Dataset({
        features: ['NAME', 'organism', 'PRODUCT_NAME'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['NAME', 'organism', 'PRODUCT_NAME'],
        num_rows: 5000
    })
})
  Train 示例:
    {'NAME': 'MSHB_MYCVP', 'organism': 'Mycolicibacterium vanbaalenii (strain DSM 7251 / JCM 13017 / BCRC 16820 / KCTC 9966 / NRRL B-24157 / PYR-1)', 'PRODUCT_NAME': '1D-myo-inositol 2-acetamido-2-deoxy-alpha-D-glucopyranoside deacetylase'}


In [ ]:
# ============ 第二步：为每个数据集设计 Instruction Prompt ============

# 定义三层级的instruction prompt函数

# 数据集1: 只有 NAME 和 PRODUCT_NAME
def format_instruction_level1(sample):
    """
    Level 1: 基础翻译任务
    输入: UniProt ID
    输出: 蛋白质名称
    """
    instruction = "Predict the protein name based on the UniProt ID."
    input_text = sample["NAME"]
    output_text = sample["PRODUCT_NAME"]
    
    prompt = f"""<s>[INST] {instruction}

UniProt ID: {input_text} [/INST]

Protein Name: {output_text}</s>"""
    
    return {
        "text": prompt,
        "instruction": instruction,
        "input": input_text,
        "output": output_text
    }

# 数据集2: NAME, organism, PRODUCT_NAME
def format_instruction_level2(sample):
    """
    Level 2: 加入生物体信息
    输入: UniProt ID + 生物体
    输出: 蛋白质名称
    """
    instruction = "Predict the protein name based on the UniProt ID and the organism."
    input_text = f"UniProt ID: {sample['NAME']}\nOrganism: {sample['organism']}"
    output_text = sample["PRODUCT_NAME"]
    
    prompt = f"""<s>[INST] {instruction}

{input_text} [/INST]

Protein Name: {output_text}</s>"""
    
    return {
        "text": prompt,
        "instruction": instruction,
        "input": input_text,
        "output": output_text
    }

# 数据集3: NAME, organism, comment, PRODUCT_NAME
def format_instruction_level3(sample):
    """
    Level 3: 加入注释信息
    输入: UniProt ID + 生物体 + 注释
    输出: 蛋白质名称
    """
    instruction = "Predict the protein name based on the UniProt ID, organism, and comments."
    comment_text = sample.get("comment", "No comment") if sample.get("comment") else "No comment"
    input_text = f"UniProt ID: {sample['NAME']}\nOrganism: {sample['organism']}\nComment: {comment_text}"
    output_text = sample["PRODUCT_NAME"]
    
    prompt = f"""<s>[INST] {instruction}

{input_text} [/INST]

Protein Name: {output_text}</s>"""
    
    return {
        "text": prompt,
        "instruction": instruction,
        "input": input_text,
        "output": output_text
    }

# 为三个数据集应用格式化函数
print("应用instruction格式化...")

dataset_dict_1_formatted = dataset_dict_1.map(
    format_instruction_level1,
    remove_columns=dataset_dict_1["train"].column_names
)
print(f"Level 1 Dataset formatted")

dataset_dict_2_formatted = dataset_dict_2.map(
    format_instruction_level2,
    remove_columns=dataset_dict_2["train"].column_names
)
print(f"Level 2 Dataset formatted")

dataset_dict_3_formatted = dataset_dict_3.map(
    format_instruction_level3,
    remove_columns=dataset_dict_3["train"].column_names
)
print(f"Level 3 Dataset formatted")

# 查看示例
print("\n=== Level 1 Example ===")
print(dataset_dict_1_formatted["train"][0]["text"])
print("\n=== Level 2 Example ===")
print(dataset_dict_2_formatted["train"][0]["text"])
print("\n=== Level 3 Example ===")
print(dataset_dict_3_formatted["train"][0]["text"])


In [ ]:
# ============ 第三步：使用 Unsloth 微调 Llama-3.1-8B ============

# 安装 unsloth 和依赖
# pip install unsloth[colab-new] @git+https://github.com/unslothai/unsloth.git
# 或 pip install unsloth[a100]

from unsloth import FastLanguageModel
import torch
from transformers import TrainingArguments, TextIteratorPatchingTrainer
from trl import SFTTrainer

# 配置参数（针对单个 A100）
MODEL_NAME = "unsloth/llama-3.1-8b"  # 或 meta-llama/Llama-3.1-8B
MAX_SEQ_LENGTH = 512
LOAD_IN_4BIT = True  # A100 也可以用 4bit 量化，更省显存
DTYPE = torch.float16
OUTPUT_DIR = "./sft_output_unsloth"

print(f"使用 Unsloth 加载模型: {MODEL_NAME}")

# 使用 Unsloth FastLanguageModel 加载（极速训练）
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

# 配置 LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth 的梯度检查点
    use_rslora=True,  # 使用 RSLora
    target_modules=["q_proj", "v_proj"],
)

# 验证可训练参数
model.print_trainable_parameters()


In [ ]:
# 数据预处理函数（Unsloth 优化版）
def formatting_func(example):
    """格式化训练数据"""
    return {"text": example["text"]}

# 训练配置（针对单个 A100）
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,  
    gradient_accumulation_steps=2,
    warmup_steps=50,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    save_strategy="steps",
    save_steps=500,
    eval_strategy="steps",
    eval_steps=500,
    load_best_model_at_end=True,
    save_total_limit=3,
    report_to=["tensorboard"],
    max_grad_norm=1.0,
)

# 训练函数
def train_level_unsloth(dataset, level_name, model, tokenizer, training_args):
    """使用 Unsloth SFTTrainer 训练"""
    print(f"\n{'='*60}")
    print(f"开始训练 Level {level_name}")
    print(f"{'='*60}")
    
    # 创建 SFTTrainer
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset["train"],
        eval_dataset=dataset["dev"],
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        args=training_args,
        packing=False,  # 设为 True 可提速但可能降低准确率
    )
    
    # 开始训练
    print(f"开始训练 Level {level_name}...")
    train_result = trainer.train()
    
    # 保存模型
    model_output_dir = f"{OUTPUT_DIR}/level_{level_name}"
    trainer.save_model(model_output_dir)
    print(f"\n✓ Level {level_name} 模型已保存到: {model_output_dir}")
    
    # 评估
    print(f"\n评估 Level {level_name}...")
    eval_result = trainer.evaluate()
    print(f"评估结果:")
    for key, value in eval_result.items():
        print(f"  {key}: {value:.4f}")
    
    return trainer, train_result, eval_result

# 使用示例（取消注释下面的代码来训练）
# trainer_1, result_1, eval_1 = train_level_unsloth(
#     dataset_dict_1_formatted, "1", model, tokenizer, training_args
# )

print("✓ 训练函数已准备。取消注释下面代码来开始训练:")
print("  train_level_unsloth(dataset_dict_1_formatted, '1', model, tokenizer, training_args)")
print("  train_level_unsloth(dataset_dict_2_formatted, '2', model, tokenizer, training_args)")
print("  train_level_unsloth(dataset_dict_3_formatted, '3', model, tokenizer, training_args)")
